<!-- notebook-header -->
# Estatistica Descritiva e Exploratoria

**Modulo:** 01 - Estatistica  
**Tipo:** Aula com exercicios guiados e solucoes executaveis  
**Descricao:** Medidas de tendencia, dispersao, outliers, visualizacoes e analise bivariada.


# 1.1 Estatistica Descritiva e Exploratoria

**Tempo estimado:** 8-10 horas
**Pre-requisitos:** 0.6 (Probabilidade), 0.8 (Otimizacao - conceito de funcao de custo)
**Proximo modulo:** 1.2 (Estatistica Inferencial)

---

## Indice

1. Introducao: Descritiva vs Inferencial
2. Medidas de Tendencia Central
3. Medidas de Dispersao
4. Medidas de Forma (Skewness e Kurtosis)
5. Percentis, Quartis e Deteccao de Outliers
6. Visualizacao de Distribuicoes (Histograma, KDE, Violin)
7. Analise Bivariada: Correlacao
8. Analise Bivariada: Variaveis Categoricas
9. Pair Plots e Exploracao Multivariada
10. Resumo Automatico com pandas
11. EDA Sistematica: Dataset Titanic
12. Exercicios Praticos
13. Erros Comuns
14. Resumo e Mapa Conceitual

## Pre-requisitos e Fio Narrativo

### De onde viemos
| Conceito | Notebook | Como usamos aqui |
|----------|----------|-------------------|
| Esperanca e variancia | 0.6 | Media = estimativa de $E[X]$, desvio padrao = $\sqrt{\text{Var}(X)}$ |
| Distribuicoes (Normal, etc.) | 0.6 | QQ-plots comparam dados com distribuicao teorica |
| Correlacao e covariancia | 0.7 | Pearson $r$ eh covariancia normalizada |
| MLE | 0.7 | Media amostral eh estimador MLE de $\mu$ |
| Funcao de custo | 0.8 | Media minimiza MSE, mediana minimiza MAE |

### Para onde vamos
| Conceito daqui | Usado em | Como |
|----------------|----------|------|
| Deteccao de outliers | Feature engineering | Limpar dados antes de modelar |
| Correlacao | Selecao de features | Identificar multicolinearidade |
| Distribuicao dos dados | Escolha de modelos | Normalidade afeta qual modelo usar |
| EDA sistematica | Todo projeto de ML | Primeiro passo antes de qualquer modelo |

### Fio narrativo
Nos modulos 0.x, construimos as ferramentas matematicas: derivadas, probabilidade, otimizacao. Agora comecamos a **usar** essas ferramentas em dados reais. Estatistica descritiva eh o primeiro passo de QUALQUER projeto de ML: antes de treinar um modelo, voce precisa **entender seus dados**. Este notebook ensina a olhar para dados de forma sistematica, usando medidas numericas e visualizacoes para extrair padroes, detectar anomalias e formular hipoteses.

## Por que Estatistica Descritiva eh Fundamental em ML?

Estatistica descritiva nao eh "so olhar dados". Ela eh a base de decisoes criticas em ML:

1. **Escolha de features:** correlacoes revelam quais variaveis sao informativas (e quais sao redundantes)
2. **Pre-processamento:** deteccao de outliers e analise de distribuicao determinam como normalizar/transformar dados
3. **Escolha de modelo:** dados assimetricos precisam de tratamento diferente de dados simetricos
4. **Diagnostico de erros:** media e mediana muito diferentes sugerem outliers que podem prejudicar o treinamento
5. **Comunicacao:** visualizacoes descritivas sao como voce apresenta resultados para stakeholders

**Por que em ML:** Sem EDA (Exploratory Data Analysis), voce esta treinando modelos as cegas. A maioria dos erros em projetos de ML vem de nao ter entendido os dados antes de modelar, nao de ter escolhido o modelo errado.

**Conexao com 0.8:** A funcao de custo MSE assume que os erros sao simetricos e sem outliers. Se a EDA revelar outliers ou assimetria, voce sabe que precisa de uma funcao de custo robusta (Huber, MAE) ou pre-processar os dados.

## 1. Introducao: Estatistica Descritiva vs Inferencial

### Analogia: Pesquisa de Opiniao

Voce quer saber a altura media dos brasileiros:
- **Abordagem descritiva:** mede 100 pessoas na rua. Calcula a media dessas 100 pessoas. "A media da minha amostra eh 1.72m." Fim. Descreve o que voce viu.
- **Abordagem inferencial:** usa essas 100 medicoes para estimar a media de TAREFA DO ALUNOS os brasileiros, com intervalo de confianca: "A media nacional eh 1.72m $\pm$ 0.03m com 95% de confianca." Generaliza para alem dos dados.

### Definicoes Formais

| Conceito | Definicao | Exemplo |
|----------|-----------|---------|
| **Populacao** | Conjunto completo de interesse | Todos os brasileiros |
| **Amostra** | Subconjunto observado | 100 pessoas medidas |
| **Parametro** | Valor verdadeiro da populacao ($\mu$, $\sigma$) | Altura media real |
| **Estatistica** | Estimativa a partir da amostra ($\bar{x}$, $s$) | Media amostral 1.72m |
| **Variavel quantitativa** | Valores numericos (continuo ou discreto) | Altura, idade |
| **Variavel qualitativa** | Categorias | Sexo, cor do carro |

**Por que em ML:** Quando treinamos um modelo, nosso dataset de treino eh uma **amostra**. O desempenho no dataset de teste tenta estimar o desempenho na **populacao** (dados futuros nunca vistos). Overfitting = decorar a amostra em vez de aprender sobre a populacao.

**Conexao com 0.6:** Populacao $\leftrightarrow$ distribuicao teorica. Amostra $\leftrightarrow$ dados observados. Media amostral $\bar{x}$ $\leftrightarrow$ estimador de $E[X]$.

In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib
matplotlib.use('Agg')
import warnings
warnings.filterwarnings('ignore')

# Tentar carregar seaborn e datasets
try:
    import seaborn as sns
    from scipy import stats
    from scipy.stats import skew, kurtosis, pearsonr, spearmanr, chi2_contingency

    plt.style.use('seaborn-v0_8-darkgrid')
    sns.set_palette('husl')

    # Datasets reais
    tips = sns.load_dataset('tips')
    titanic = sns.load_dataset('titanic')
    penguins = sns.load_dataset('penguins').dropna()

    print(f"Datasets carregados:")
    print(f"  Tips:     {tips.shape[0]} linhas, {tips.shape[1]} colunas")
    print(f"  Titanic:  {titanic.shape[0]} linhas, {titanic.shape[1]} colunas")
    print(f"  Penguins: {penguins.shape[0]} linhas, {penguins.shape[1]} colunas")
    HAS_SEABORN = True
except ImportError:
    print("seaborn/scipy nao disponiveis - rode em Jupyter com pip install seaborn scipy")
    HAS_SEABORN = False

Datasets carregados:
  Tips:     244 linhas, 7 colunas
  Titanic:  891 linhas, 15 colunas
  Penguins: 333 linhas, 7 colunas


## 2. Medidas de Tendencia Central

### Analogia: Onde os Dados se Concentram?

Imagine notas de 10 alunos: [5, 6, 7, 7, 8, 8, 8, 9, 9, 10]

- **Media ($\bar{x}$):** se redistribuisse igualmente, cada um teria 7.7. Eh o "centro de gravidade"
- **Mediana:** o valor do meio quando ordenados (7.5 aqui). Robusto a outliers
- **Moda:** valor mais frequente (8). Util para categorias

Se um aluno tirasse 0, a media cairia para 7.0, mas a mediana mal mudaria (de 7.5 para 7). Por isso a mediana eh **robusta**.

### Definicoes Formais

$$\text{Media: } \bar{x} = \frac{1}{n}\sum_{i=1}^n x_i \qquad \text{Mediana: } \tilde{x} = \begin{cases} x_{(n+1)/2} & n \text{ impar} \\ \frac{x_{n/2} + x_{n/2+1}}{2} & n \text{ par}\end{cases}$$

| Medida | Vantagem | Desvantagem | Quando usar |
|--------|----------|-------------|-------------|
| Media | Usa todos os dados | Sensivel a outliers | Dados simetricos |
| Mediana | Robusta a outliers | Ignora magnitudes | Dados assimetricos |
| Moda | Funciona para categorias | Pode nao ser unica | Dados categoricos |

**Por que em ML:** A media amostral eh o estimador MLE de $\mu$ para dados normais (conexao 0.7). Quando voce calcula `np.mean(y_pred - y_true)`, esta estimando o vies do modelo.

**Conexao com 0.8:** Minimizar MSE $= \frac{1}{n}\sum(y_i - c)^2$ produz $c = \bar{x}$ (media). Minimizar MAE $= \frac{1}{n}\sum|y_i - c|$ produz $c = \tilde{x}$ (mediana). A funcao de custo determina qual "centro" o modelo busca!

In [2]:
# Média aritmética
mean_total_bill = tips['total_bill'].mean()
print(f'Média da conta total: ${mean_total_bill:.2f}')

# Média ponderada (exemplo: média ponderada da nota de um aluno)
grades = np.array([7, 8, 9])
weights = np.array([0.2, 0.3, 0.5])
weighted_mean = np.average(grades, weights=weights)
print(f'Média ponderada (exemplo): {weighted_mean:.2f}')

# Média geométrica (para taxas de crescimento)
rates = np.array([1.05, 1.10, 1.08])  # 5%, 10%, 8% crescimento
geometric_mean = stats.gmean(rates)
print(f'Média geométrica de taxas: {geometric_mean:.4f} (aprox. {(geometric_mean-1)*100:.2f}% crescimento médio)')

# Mediana (valor central)
median_total_bill = tips['total_bill'].median()
print(f'Mediana da conta total: ${median_total_bill:.2f}')

# Moda (valor mais frequente)
mode_day = tips['day'].mode()
print(f'Moda (dia mais frequente): {mode_day[0]}')

# Comparação em distribuição assimétrica
fig, axes = plt.subplots(1, 2, figsize=(14, 4))
axes[0].hist(tips['total_bill'], bins=30, edgecolor='black', alpha=0.7)
axes[0].axvline(mean_total_bill, color='red', linestyle='--', linewidth=2, label=f'Média: ${mean_total_bill:.2f}')
axes[0].axvline(median_total_bill, color='green', linestyle='--', linewidth=2, label=f'Mediana: ${median_total_bill:.2f}')
axes[0].set_xlabel('Valor da Conta ($)')
axes[0].set_ylabel('Frequência')
axes[0].set_title('Distribuição: Conta Total (Tips)')
axes[0].legend()

axes[1].hist(titanic['age'].dropna(), bins=30, edgecolor='black', alpha=0.7)
mean_age = titanic['age'].mean()
median_age = titanic['age'].median()
axes[1].axvline(mean_age, color='red', linestyle='--', linewidth=2, label=f'Média: {mean_age:.1f}')
axes[1].axvline(median_age, color='green', linestyle='--', linewidth=2, label=f'Mediana: {median_age:.1f}')
axes[1].set_xlabel('Idade (anos)')
axes[1].set_ylabel('Frequência')
axes[1].set_title('Distribuição: Idade (Titanic)')
axes[1].legend()
plt.tight_layout()
plt.show()

Média da conta total: $19.79
Média ponderada (exemplo): 8.30
Média geométrica de taxas: 1.0765 (aprox. 7.65% crescimento médio)
Mediana da conta total: $17.80
Moda (dia mais frequente): Sat


**O que observar:**
- Grafico esquerdo (conta total): a media (vermelha) esta a direita da mediana (verde), indicando assimetria positiva -- contas altas puxam a media
- Grafico direito (idade): media e mediana quase coincidentes, sugerindo distribuicao mais simetrica
- A moda do dia mais frequente revela que sabado eh o dia com mais dados

**O que concluir:**
- Quando media > mediana: assimetria positiva (cauda direita longa) -- comum em dados financeiros
- Quando media $\approx$ mediana: distribuicao aproximadamente simetrica
- Na pratica de ML: se media e mediana diferem muito, considere usar mediana (mais robusta) ou transformar os dados (log, Box-Cox)

**Conexao com 0.6:** A assimetria positiva nos valores de conta eh consistente com uma distribuicao log-normal. Se $\log(X) \sim \mathcal{N}$, entao a media geometrica pode ser mais informativa que a aritmetica.

## 3. Medidas de Dispersao

### Analogia: Previsibilidade de Restaurantes

Dois restaurantes com media de conta = R\$50:
- **Restaurante A:** contas entre R\$40-60 (desvio padrao baixo = previsivel)
- **Restaurante B:** contas entre R\$10-200 (desvio padrao alto = imprevisivel)

A media sozinha nao distingue os dois. Precisamos medir o **espalhamento** dos dados.

### Definicoes Formais

$$\text{Variancia amostral: } s^2 = \frac{1}{n-1}\sum_{i=1}^n (x_i - \bar{x})^2$$

$$\text{Desvio padrao: } s = \sqrt{s^2} \qquad \text{CV: } \frac{s}{\bar{x}} \times 100\%$$

$$\text{IQR: } Q_3 - Q_1 \qquad \text{MAD: } \text{mediana}(|x_i - \tilde{x}|)$$

| Medida | Robusta? | Unidade | Quando usar |
|--------|----------|---------|-------------|
| Variancia ($s^2$) | Nao | Quadrado da original | Calculos teoricos |
| Desvio padrao ($s$) | Nao | Mesma dos dados | Uso geral, dados normais |
| CV | Nao | Adimensional (%) | Comparar dispersao entre escalas |
| IQR | Sim | Mesma dos dados | Dados com outliers |
| MAD | Sim | Mesma dos dados | Alternativa robusta ao $s$ |

**Regra 68-95-99.7:** Em distribuicao normal, ~68% dos dados estao em $\mu \pm \sigma$, ~95% em $\mu \pm 2\sigma$, ~99.7% em $\mu \pm 3\sigma$.

**Por que em ML:** Desvio padrao eh central em normalizacao de features (StandardScaler: $x' = (x - \mu)/\sigma$). Features com alta dispersao dominam modelos baseados em distancia (KNN, SVM). Normalizar garante que todas as features contribuam igualmente.

**Conexao com 0.6:** Variancia amostral $s^2$ estima $\text{Var}(X)$. O divisor $n-1$ (correcao de Bessel) garante estimador nao-viesado -- lembre que MLE de $\sigma^2$ usa $n$ e eh viesado (0.7).

In [3]:
# Variância (média dos desvios ao quadrado)
var_total_bill = tips['total_bill'].var(ddof=1)  # ddof=1 para amostra
print(f'Variância (amostra): {var_total_bill:.2f}')

# Desvio padrão (raiz quadrada da variância)
std_total_bill = tips['total_bill'].std(ddof=1)
print(f'Desvio padrão: ${std_total_bill:.2f}')
print(f'Coeficiente de variação: {(std_total_bill/mean_total_bill)*100:.2f}%')

# Intervalo interquartil (IQR)
q1 = tips['total_bill'].quantile(0.25)
q3 = tips['total_bill'].quantile(0.75)
iqr = q3 - q1
print(f'Q1: ${q1:.2f}, Q3: ${q3:.2f}, IQR: ${iqr:.2f}')

# Range
range_val = tips['total_bill'].max() - tips['total_bill'].min()
print(f'Range: ${range_val:.2f}')

# MAD (Median Absolute Deviation)
mad = np.median(np.abs(tips['total_bill'] - median_total_bill))
print(f'MAD (Desvio Absoluto Mediano): ${mad:.2f}')

# Visualização
fig, axes = plt.subplots(1, 2, figsize=(14, 4))

# Boxplot
axes[0].boxplot([tips['total_bill'], titanic['age'].dropna()], labels=['Total Bill', 'Age'])
axes[0].set_ylabel('Valor')
axes[0].set_title('Boxplot: Dispersão de Dados')
axes[0].grid(True, alpha=0.3)

# Distribuição com desvios padrão
ages_clean = titanic['age'].dropna()
axes[1].hist(ages_clean, bins=30, alpha=0.7, edgecolor='black', density=True, label='Dados')
mu, sigma = ages_clean.mean(), ages_clean.std()
x = np.linspace(mu - 4*sigma, mu + 4*sigma, 100)
axes[1].plot(x, stats.norm.pdf(x, mu, sigma), 'r-', linewidth=2, label='Normal (μ, σ)')
axes[1].axvline(mu, color='red', linestyle='--', alpha=0.5, label='μ')
axes[1].axvline(mu - sigma, color='orange', linestyle='--', alpha=0.5)
axes[1].axvline(mu + sigma, color='orange', linestyle='--', alpha=0.5, label='±σ')
axes[1].set_xlabel('Idade')
axes[1].set_ylabel('Densidade')
axes[1].set_title('Distribuição Normal com Desvios Padrão')
axes[1].legend()
plt.tight_layout()
plt.show()

Variância (amostra): 79.25
Desvio padrão: $8.90
Coeficiente de variação: 44.99%
Q1: $13.35, Q3: $24.13, IQR: $10.78
Range: $47.74
MAD (Desvio Absoluto Mediano): $5.03


**O que observar:**
- Boxplot (esquerdo): a caixa mostra IQR (50% central dos dados), bigodes mostram faixa sem outliers, pontos sao outliers
- Distribuicao com desvio padrao (direito): as linhas laranjas ($\pm\sigma$) capturam a maior parte dos dados
- O MAD (desvio absoluto mediano) eh menor que o desvio padrao, como esperado para dados com outliers

**O que concluir:**
- CV (coeficiente de variacao) permite comparar dispersao entre variaveis com escalas diferentes
- IQR e MAD sao alternativas robustas quando ha outliers -- nao sao "puxadas" por valores extremos
- Na pratica: use `ddof=1` (divisao por $n-1$) para amostra, `ddof=0` para populacao

**Conexao com 0.7:** O desvio padrao do erro ($\sigma_\epsilon$) em regressao determina a largura dos intervalos de confianca. Quanto maior $\sigma$, mais incertos sao os intervalos.

## 4. Medidas de Forma: Skewness e Kurtosis

### Analogia: Formato da Montanha

Imagine que a distribuicao eh uma montanha:
- **Skewness (assimetria):** a montanha esta inclinada para um lado? Cauda longa a direita (positiva) ou esquerda (negativa)?
- **Kurtosis (curtose):** a montanha tem pico agudo com caudas pesadas (leptocurtica) ou pico achatado com caudas leves (platicurtica)?

### Definicoes Formais

$$\text{Skewness: } \gamma_1 = \frac{1}{n}\sum\left(\frac{x_i - \bar{x}}{s}\right)^3 \qquad \text{Kurtosis: } \gamma_2 = \frac{1}{n}\sum\left(\frac{x_i - \bar{x}}{s}\right)^4 - 3$$

| Valor | Skewness | Kurtosis (excesso) |
|-------|----------|-------------------|
| $= 0$ | Simetrica | Normal (mesocurtica) |
| $> 0$ | Cauda direita | Caudas pesadas (leptocurtica) |
| $< 0$ | Cauda esquerda | Caudas leves (platicurtica) |

**QQ-Plot:** compara quantis dos dados com quantis de uma distribuicao teorica (geralmente Normal). Se os pontos seguem a diagonal, os dados sao aproximadamente normais.

**Por que em ML:** Muitos modelos assumem normalidade (regressao linear, LDA). Skewness alta sugere necessidade de transformacao (log, Box-Cox). Kurtosis alta indica caudas pesadas -- outliers mais provaveis, o que afeta MSE.

**Conexao com 0.6:** Skewness e kurtosis sao o 3o e 4o momentos padronizados. Os dois primeiros (media e variancia) nao capturam toda a informacao da distribuicao.

In [4]:
# Skewness (assimetria)
skewness_bills = skew(tips['total_bill'])
skewness_age = skew(titanic['age'].dropna())
print(f'Skewness - Total Bill: {skewness_bills:.3f} (assimétria positiva/direita)')
print(f'Skewness - Age: {skewness_age:.3f}')
print('\nInterpretação: skewness > 0.5 = assimétrica positiva; < -0.5 = negativa')

# Kurtosis (caudas pesadas?)
kurtosis_bills = kurtosis(tips['total_bill'])
kurtosis_age = kurtosis(titanic['age'].dropna())
print(f'\nKurtosis - Total Bill: {kurtosis_bills:.3f}')
print(f'Kurtosis - Age: {kurtosis_age:.3f}')
print('Kurtosis > 3 = caudas pesadas (leptocúrtica); < 3 = caudas leves (platicúrtica)')

# Visualização
fig, axes = plt.subplots(2, 3, figsize=(15, 8))

# Exemplos de diferentes skewness
data_right = np.random.beta(2, 5, 1000)  # Assimétrica à direita
data_left = np.random.beta(5, 2, 1000)   # Assimétrica à esquerda
data_normal = np.random.normal(0, 1, 1000)  # Normal

axes[0, 0].hist(data_right, bins=30, edgecolor='black', alpha=0.7)
axes[0, 0].set_title(f'Skewness Positiva ({skew(data_right):.2f})')
axes[0, 0].axvline(np.mean(data_right), color='red', linestyle='--', label='Média')
axes[0, 0].axvline(np.median(data_right), color='green', linestyle='--', label='Mediana')
axes[0, 0].legend()

axes[0, 1].hist(data_left, bins=30, edgecolor='black', alpha=0.7)
axes[0, 1].set_title(f'Skewness Negativa ({skew(data_left):.2f})')
axes[0, 1].axvline(np.mean(data_left), color='red', linestyle='--', label='Média')
axes[0, 1].axvline(np.median(data_left), color='green', linestyle='--', label='Mediana')
axes[0, 1].legend()

axes[0, 2].hist(data_normal, bins=30, edgecolor='black', alpha=0.7)
axes[0, 2].set_title(f'Simétrica ({skew(data_normal):.2f})')
axes[0, 2].axvline(np.mean(data_normal), color='red', linestyle='--', label='Média')
axes[0, 2].axvline(np.median(data_normal), color='green', linestyle='--', label='Mediana')
axes[0, 2].legend()

# QQ-plots
# stats.probplot(data_right, dist='norm', plot=axes[1, 0])
axes[1, 0].set_title('QQ-plot: Dados Assimétricos')

# stats.probplot(data_normal, dist='norm', plot=axes[1, 1])
axes[1, 1].set_title('QQ-plot: Dados Normais')

# stats.probplot(titanic['age'].dropna(), dist='norm', plot=axes[1, 2])
axes[1, 2].set_title('QQ-plot: Idade (Titanic)')

plt.tight_layout()
plt.show()

Skewness - Total Bill: 1.126 (assimétria positiva/direita)
Skewness - Age: 0.388

Interpretação: skewness > 0.5 = assimétrica positiva; < -0.5 = negativa

Kurtosis - Total Bill: 1.169
Kurtosis - Age: 0.169
Kurtosis > 3 = caudas pesadas (leptocúrtica); < 3 = caudas leves (platicúrtica)


**O que observar:**
- Linha superior: distribuicoes com skewness positiva (cauda direita), negativa (cauda esquerda) e simetrica
- Note como media (vermelha) e mediana (verde) se separam conforme a assimetria aumenta
- QQ-plots: dados normais seguem a diagonal; dados assimetricos curvam nas extremidades

**O que concluir:**
- $|\text{skewness}| > 0.5$: assimetria moderada que pode precisar de tratamento
- $|\text{skewness}| > 1$: assimetria severa -- considere transformacao log
- QQ-plot eh a ferramenta mais visual para checar normalidade
- Se os pontos do QQ-plot curvam para cima na extremidade direita: cauda direita pesada

**Conexao com 0.7:** Se voce assume normalidade para MLE mas os dados sao assimetricos, sua estimativa de $\mu$ (media) sera puxada pelos outliers. A mediana (estimador MLE sob distribuicao Laplaciana) seria mais apropriada.

## 5. Percentis, Quartis e Deteccao de Outliers

### Analogia: Notas da Turma

Se voce esta no percentil 90, significa que 90% da turma tirou nota menor ou igual a sua. Quartis dividem a turma em 4 grupos iguais:
- Q1 (25o percentil): abaixo de 25% dos alunos
- Q2 (mediana): metade da turma
- Q3 (75o percentil): acima de 75% dos alunos

### Metodos de Deteccao de Outliers

| Metodo | Regra | Robusto? | Premissa |
|--------|-------|----------|----------|
| IQR | $x < Q_1 - 1.5 \cdot \text{IQR}$ ou $x > Q_3 + 1.5 \cdot \text{IQR}$ | Sim | Nenhuma |
| Z-score | $|z| > 3$ onde $z = (x - \bar{x})/s$ | Nao | Normalidade aprox. |
| MAD | $|x - \tilde{x}| > 3 \cdot \text{MAD}$ | Sim | Nenhuma |

**Por que em ML:** Outliers podem destruir modelos baseados em MSE (um outlier extremo domina o gradiente). Decidir o que fazer com outliers (remover, transformar, usar modelo robusto) eh uma das decisoes mais importantes de pre-processamento.

**Conexao com 0.8:** Se outliers estao presentes e voce usa MSE, considere trocar para Huber loss ou MAE (funcoes de custo robustas do 0.8).

In [5]:
# Percentis
percentiles = [10, 25, 50, 75, 90, 95, 99]
for p in percentiles:
    val = np.percentile(tips['total_bill'], p)
    print(f'{p}º percentil: ${val:.2f}')

# Método IQR para detectar outliers
Q1 = tips['total_bill'].quantile(0.25)
Q3 = tips['total_bill'].quantile(0.75)
IQR = Q3 - Q1
lower_bound = Q1 - 1.5 * IQR
upper_bound = Q3 + 1.5 * IQR

outliers_iqr = tips[(tips['total_bill'] < lower_bound) | (tips['total_bill'] > upper_bound)]
print(f'\nOutliers (IQR): {len(outliers_iqr)} observações')
print(f'Limites: [{lower_bound:.2f}, {upper_bound:.2f}]')

# Método Z-score
z_scores = np.abs(stats.zscore(tips['total_bill']))
outliers_zscore = tips[z_scores > 3]
print(f'Outliers (Z-score > 3): {len(outliers_zscore)} observações')

# Visualização
fig, axes = plt.subplots(1, 2, figsize=(14, 4))

# Boxplot com outliers
axes[0].boxplot(tips['total_bill'], vert=True)
axes[0].scatter([1]*len(outliers_iqr), outliers_iqr['total_bill'], color='red', s=100, zorder=5, label='Outliers')
axes[0].set_ylabel('Valor da Conta ($)')
axes[0].set_title('Boxplot com Outliers (IQR)')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# Scatter: índice vs valor
axes[1].scatter(range(len(tips)), tips['total_bill'], alpha=0.6, label='Dados')
axes[1].scatter(outliers_iqr.index, outliers_iqr['total_bill'], color='red', s=100, label='Outliers')
axes[1].axhline(upper_bound, color='orange', linestyle='--', label=f'Limite superior: ${upper_bound:.2f}')
axes[1].axhline(lower_bound, color='orange', linestyle='--', label=f'Limite inferior: ${lower_bound:.2f}')
axes[1].set_xlabel('Índice')
axes[1].set_ylabel('Valor da Conta ($)')
axes[1].set_title('Detecção de Outliers (IQR)')
axes[1].legend()
plt.tight_layout()
plt.show()

10º percentil: $10.34
25º percentil: $13.35
50º percentil: $17.80
75º percentil: $24.13
90º percentil: $32.24
95º percentil: $38.06
99º percentil: $48.23

Outliers (IQR): 9 observações
Limites: [-2.82, 40.30]
Outliers (Z-score > 3): 4 observações


**O que observar:**
- Boxplot (esquerdo): os pontos vermelhos acima do bigode superior sao outliers pelo criterio IQR
- Scatter (direito): as linhas laranjas horizontais delimitam a faixa "normal"; pontos acima sao outliers
- Z-score detectou menos outliers que IQR porque Z-score assume normalidade

**O que concluir:**
- IQR eh o metodo mais usado na pratica por nao assumir normalidade
- Outlier nao significa "dado errado" -- pode ser informacao valiosa (conta de grupo grande no restaurante)
- Antes de remover outliers, investigue a causa: erro de medicao? Subpopulacao diferente? Dado valido?
- Em ML, considere: (a) remover, (b) transformar (log), (c) usar modelo robusto, (d) winsorizar (limitar ao percentil 95)

**Conexao com 0.6:** Pela regra 68-95-99.7, em dados normais esperamos ~0.3% de dados alem de $3\sigma$. Se ha mais que isso, a distribuicao nao eh normal ou ha outliers genuinos.

## 6. Visualizacao de Distribuicoes

### Tipos de Graficos e Quando Usar

| Grafico | Mostra | Quando usar |
|---------|--------|-------------|
| Histograma | Forma da distribuicao | Sempre (primeiro olhar) |
| KDE | Estimativa suave de densidade | Comparar distribuicoes sobrepostas |
| Boxplot | Mediana, IQR, outliers | Comparar grupos |
| Violin | Distribuicao + densidade | Comparar grupos com mais detalhe |
| Strip | Dados individuais | Poucos dados (n < 100) |

### Escolha do Numero de Bins (Histograma)

A escolha de bins afeta a impressao visual:
- Poucos bins: perde detalhes (over-smoothing)
- Muitos bins: ruido domina (under-smoothing)
- Regra de Sturges: $k = 1 + \log_2(n)$
- Regra de Scott: $h = 3.5 s / n^{1/3}$

**Por que em ML:** Visualizar distribuicoes eh essencial para decidir transformacoes (log para dados assimetricos), detectar bimodalidade (pode indicar subpopulacoes), e verificar premissas de modelos.

**Conexao com 0.6:** KDE (Kernel Density Estimation) eh uma estimativa nao-parametrica da PDF. Cada ponto gera um kernel (geralmente gaussiano) e a soma deles forma a curva suave.

In [6]:
# Múltiplas visualizações de distribuição
fig, axes = plt.subplots(2, 3, figsize=(16, 8))

# Histograma simples
axes[0, 0].hist(tips['total_bill'], bins=20, edgecolor='black', alpha=0.7, color='skyblue')
axes[0, 0].set_xlabel('Valor da Conta ($)')
axes[0, 0].set_ylabel('Frequência')
axes[0, 0].set_title('Histograma: Distribuição Simples')

# KDE Plot (Kernel Density Estimation)
sns.kdeplot(data=tips, x='total_bill', ax=axes[0, 1], fill=True, color='lightblue')
axes[0, 1].set_xlabel('Valor da Conta ($)')
axes[0, 1].set_title('KDE Plot: Estimativa de Densidade')

# Histograma + KDE
axes[0, 2].hist(tips['total_bill'], bins=20, edgecolor='black', alpha=0.5, density=True, color='skyblue')
sns.kdeplot(data=tips, x='total_bill', ax=axes[0, 2], color='darkblue', linewidth=2)
axes[0, 2].set_xlabel('Valor da Conta ($)')
axes[0, 2].set_title('Histograma + KDE')

# Violin plot (distribuição por grupo)
sns.violinplot(data=tips, x='day', y='total_bill', ax=axes[1, 0])
axes[1, 0].set_title('Violin Plot: Conta por Dia da Semana')
axes[1, 0].set_ylabel('Valor da Conta ($)')

# Violin plot com pontos
sns.violinplot(data=tips, x='sex', y='total_bill', ax=axes[1, 1], inner=None, alpha=0.6)
sns.stripplot(data=tips, x='sex', y='total_bill', ax=axes[1, 1], color='black', alpha=0.3, size=4)
axes[1, 1].set_title('Violin Plot com Dados: Conta por Sexo')
axes[1, 1].set_ylabel('Valor da Conta ($)')

# Distribuição por múltiplos grupos
sns.violinplot(data=tips, x='day', y='total_bill', hue='sex', ax=axes[1, 2], split=False)
axes[1, 2].set_title('Violin Plot: Conta por Dia e Sexo')
axes[1, 2].set_ylabel('Valor da Conta ($)')

plt.tight_layout()
plt.show()

**O que observar:**
- Histograma vs KDE: o histograma mostra contagens em bins, o KDE suaviza para uma curva continua
- Histograma + KDE juntos: a melhor visualizacao para entender a forma da distribuicao
- Violin plots: mostram a distribuicao completa (nao so quartis como boxplot)
- Violin split por grupo: permite comparar distribuicoes lado a lado

**O que concluir:**
- Use KDE quando quiser comparar multiplas distribuicoes no mesmo eixo (menos "ruidoso" que histogramas sobrepostos)
- Violin plots sao superiores a boxplots quando voce quer ver bimodalidade ou assimetria
- Adicionar strip/swarm plot sobre violin mostra pontos individuais -- util para detectar clusters

**Conexao com 0.6:** O KDE com kernel gaussiano eh basicamente somar $n$ distribuicoes $\mathcal{N}(x_i, h^2)$, onde $h$ eh a bandwidth. Bandwidth grande = suave (underfitting), pequena = ruidoso (overfitting).

## 7. Analise Bivariada: Correlacao

### Analogia: Dois Dançarinos

Correlacao mede se duas variaveis "dancam juntas":
- $r = +1$: dancam em perfeita sincronia (quando um sobe, o outro sobe)
- $r = -1$: dancam em oposicao perfeita (quando um sobe, o outro desce)
- $r = 0$: dancam independentemente (nao ha padrao)

### Definicao Formal

$$r_{\text{Pearson}} = \frac{\sum(x_i - \bar{x})(y_i - \bar{y})}{\sqrt{\sum(x_i - \bar{x})^2 \cdot \sum(y_i - \bar{y})^2}} = \frac{\text{Cov}(X,Y)}{s_X \cdot s_Y}$$

| Tipo | Mede | Premissa |
|------|------|----------|
| Pearson | Relacao linear | Normalidade, linearidade |
| Spearman | Relacao monotonica | Nenhuma (baseado em rankings) |

**CUIDADO:** Correlacao $\neq$ causalidade! Vendas de sorvete correlacionam com afogamentos, mas a causa comum eh o calor.

**Por que em ML:** Correlacao alta entre features indica **multicolinearidade**, que prejudica regressao linear (coeficientes instaveis). Na selecao de features, remover uma de cada par com $|r| > 0.8$ eh pratica comum.

**Conexao com 0.7:** Pearson $r$ eh a covariancia normalizada: $r = \text{Cov}(X,Y) / (\sigma_X \sigma_Y)$. A covariancia foi definida formalmente em 0.7 para distribuicoes conjuntas.

In [7]:
# Correlação de Pearson
corr_pearson = tips['total_bill'].corr(tips['tip'])
print(f'Correlação de Pearson (conta vs gorjeta): {corr_pearson:.3f}')

# Teste de significância
from scipy.stats import pearsonr
corr, p_value = pearsonr(tips['total_bill'], tips['tip'])
print(f'P-value: {p_value:.2e} (significante: {p_value < 0.05})')

# Correlação de Spearman (não-paramétrica)
from scipy.stats import spearmanr
corr_spearman, p_value_spearman = spearmanr(tips['total_bill'], tips['tip'])
print(f'\nCorrelação de Spearman: {corr_spearman:.3f}')
print(f'P-value: {p_value_spearman:.2e}')

# Matriz de correlação
numeric_cols = tips.select_dtypes(include=[np.number]).columns
corr_matrix = tips[numeric_cols].corr()
print('\nMatriz de Correlação (Tips):')
print(corr_matrix)

# Visualização
fig, axes = plt.subplots(1, 3, figsize=(16, 4))

# Scatter plot
axes[0].scatter(tips['total_bill'], tips['tip'], alpha=0.6, s=50)
z = np.polyfit(tips['total_bill'], tips['tip'], 1)
p = np.poly1d(z)
axes[0].plot(tips['total_bill'], p(tips['total_bill']), 'r--', linewidth=2, label=f'Ajuste linear (r={corr_pearson:.3f})')
axes[0].set_xlabel('Valor da Conta ($)')
axes[0].set_ylabel('Gorjeta ($)')
axes[0].set_title('Scatter Plot: Correlação Positiva')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# Hexbin plot (densidade)
hexbin = axes[1].hexbin(tips['total_bill'], tips['tip'], gridsize=15, cmap='YlOrRd', mincnt=1)
axes[1].set_xlabel('Valor da Conta ($)')
axes[1].set_ylabel('Gorjeta ($)')
axes[1].set_title('Hexbin Plot: Densidade de Pontos')
plt.colorbar(hexbin, ax=axes[1])

# Heatmap de correlação
sns.heatmap(corr_matrix, annot=True, cmap='coolwarm', center=0, square=True, ax=axes[2], 
            cbar_kws={'label': 'Correlação'})
axes[2].set_title('Heatmap: Matriz de Correlação')

plt.tight_layout()
plt.show()

Correlação de Pearson (conta vs gorjeta): 0.676
P-value: 6.69e-34 (significante: True)

Correlação de Spearman: 0.679
P-value: 2.50e-34

Matriz de Correlação (Tips):
            total_bill       tip      size
total_bill    1.000000  0.675734  0.598315
tip           0.675734  1.000000  0.489299
size          0.598315  0.489299  1.000000


**O que observar:**
- Scatter plot: relacao claramente linear positiva entre valor da conta e gorjeta ($r \approx 0.68$)
- Hexbin: mostra onde os dados se concentram (util para muitos pontos sobrepostos)
- Heatmap: visao geral de todas as correlacoes; vermelho = positiva, azul = negativa

**O que concluir:**
- Conta total e gorjeta tem correlacao positiva forte: contas maiores geram gorjetas maiores
- O p-valor muito pequeno confirma que a correlacao nao eh por acaso
- A correlacao mais fraca entre tamanho do grupo e gorjeta sugere que a relacao nao eh tao forte
- Sempre verifique o scatter plot -- um $r$ alto pode esconder padroes nao-lineares

**Conexao com 0.7:** O teste de significancia da correlacao verifica $H_0: \rho = 0$ usando estatistica $t = r\sqrt{(n-2)/(1-r^2)}$ com $n-2$ graus de liberdade. Isso sera formalizado em 1.2 (Estatistica Inferencial).

## 8. Analise Bivariada: Variaveis Categoricas

### Tabela de Contingencia e Teste Qui-Quadrado

Para duas variaveis categoricas, a ferramenta principal eh a **tabela de contingencia**: contagem de ocorrencias para cada combinacao de categorias.

O **teste qui-quadrado** ($\chi^2$) verifica se as duas variaveis sao independentes:
- $H_0$: as variaveis sao independentes (distribuicao observada = esperada)
- $H_1$: existe associacao entre elas

$$\chi^2 = \sum \frac{(O_{ij} - E_{ij})^2}{E_{ij}}$$

onde $O$ = frequencia observada, $E$ = frequencia esperada sob independencia.

**Por que em ML:** Testes de associacao entre variaveis categoricas ajudam na selecao de features categoricas. Se uma feature categorica nao tem associacao com o target (p-valor alto), ela provavelmente nao eh informativa.

**Conexao com 0.6:** O teste qui-quadrado usa a distribuicao $\chi^2$, que eh a soma de $k$ normais ao quadrado. Os graus de liberdade sao $(r-1)(c-1)$ onde $r$ = linhas, $c$ = colunas.

In [8]:
# Tabela de contingência
from scipy.stats import chi2_contingency

contingency_table = pd.crosstab(tips['sex'], tips['day'])
print('Tabela de Contingência (Sexo vs Dia):')
print(contingency_table)

# Teste qui-quadrado
chi2, p_value, dof, expected = chi2_contingency(contingency_table)
print(f'\nTeste Qui-Quadrado:')
print(f'Chi² = {chi2:.3f}, p-value = {p_value:.3f}')
print(f'Significante: {p_value < 0.05}')

# Propósito: frequência relativa
contingency_relative = pd.crosstab(tips['sex'], tips['day'], normalize='index')
print('\nTabela de Frequência Relativa (% por sexo):')
print((contingency_relative * 100).round(1))

# Visualização
fig, axes = plt.subplots(1, 3, figsize=(16, 4))

# Gráfico de barras agrupado
contingency_table.plot(kind='bar', ax=axes[0], alpha=0.8)
axes[0].set_xlabel('Sexo')
axes[0].set_ylabel('Contagem')
axes[0].set_title('Gráfico de Barras: Sexo vs Dia (Contagem)')
axes[0].legend(title='Dia')
axes[0].grid(True, alpha=0.3, axis='y')

# Heatmap da tabela de contingência
sns.heatmap(contingency_table, annot=True, fmt='d', cmap='Blues', ax=axes[1], cbar_kws={'label': 'Contagem'})
axes[1].set_title('Heatmap: Tabela de Contingência')
axes[1].set_ylabel('Sexo')
axes[1].set_xlabel('Dia')

# Gráfico de barras empilhadas (100%)
contingency_relative.plot(kind='bar', stacked=True, ax=axes[2], alpha=0.8)
axes[2].set_xlabel('Sexo')
axes[2].set_ylabel('Proporção')
axes[2].set_title('Gráfico de Barras: Distribuição Relativa')
axes[2].legend(title='Dia')
axes[2].grid(True, alpha=0.3, axis='y')

plt.tight_layout()
plt.show()

Tabela de Contingência (Sexo vs Dia):
day     Thur  Fri  Sat  Sun
sex                        
Male      30   10   59   58
Female    32    9   28   18

Teste Qui-Quadrado:
Chi² = 13.222, p-value = 0.004
Significante: True

Tabela de Frequência Relativa (% por sexo):
day     Thur   Fri   Sat   Sun
sex                           
Male    19.1   6.4  37.6  36.9
Female  36.8  10.3  32.2  20.7


**O que observar:**
- Grafico de barras agrupado: permite comparar contagens absolutas entre grupos
- Heatmap: mostra a estrutura da tabela de contingencia visualmente
- Barras empilhadas 100%: mostra proporcoes relativas (mais facil de comparar que contagens absolutas)

**O que concluir:**
- O teste qui-quadrado determina se ha associacao estatisticamente significativa entre as variaveis
- Se p-valor < 0.05: rejeita independencia (as variaveis estao associadas)
- Frequencias relativas (por linha ou coluna) sao mais informativas que absolutas para comparacao
- Em ML, use tabelas de contingencia para entender a relacao entre features categoricas e o target

**Conexao com 0.7:** O teste qui-quadrado eh essencialmente um teste de likelihood ratio: compara a verossimilhanca sob independencia ($H_0$) com a verossimilhanca dos dados observados.

## 9. Pair Plots e Exploracao Multivariada

### Por que Pair Plots?

Com $d$ variaveis, existem $\binom{d}{2} = d(d-1)/2$ pares possiveis. O pair plot mostra TAREFA DO ALUNOS os scatter plots de uma vez, com distribuicoes na diagonal.

Colorir por uma variavel categorica (hue) revela se subgrupos tem padroes diferentes.

**Por que em ML:** Pair plots sao a forma mais rapida de detectar: (a) correlacoes entre features, (b) clusters naturais nos dados, (c) features que separam bem as classes (para classificacao). Eh o "raio-X" do dataset.

**Conexao com 0.7:** Para dados multivariados, a distribuicao conjunta $P(X_1, ..., X_d)$ contem toda a informacao. Pair plots mostram as distribuicoes marginais (diagonal) e relacoes bivariadas (off-diagonal).

In [9]:
# Pair plot: todos os relacionamentos entre variáveis numéricas
# fig = plt.pairplot(tips[['total_bill', 'tip', 'size']], diag_kind='kde', plot_kws={'alpha': 0.6})
fig.suptitle('Pair Plot: Múltiplas Variáveis (Tips Dataset)', y=1.01)
plt.show()

# Pair plot com cores por categoria
# fig = plt.pairplot(tips[['total_bill', 'tip', 'size', 'sex']], hue='sex', diag_kind='kde', plot_kws={'alpha': 0.6})
fig.suptitle('Pair Plot Colorido: Por Sexo (Tips Dataset)', y=1.01)
plt.show()

**O que observar:**
- Na diagonal: KDE mostra a distribuicao de cada variavel individualmente
- Fora da diagonal: scatter plots mostram relacoes entre pares de variaveis
- Colorido por sexo: revela se padroes diferem entre subgrupos
- Clusters visiveis nos scatter plots podem indicar subpopulacoes naturais

**O que concluir:**
- Pair plots sao essenciais como primeiro passo de EDA para dados multivariados
- Quando coloridos por uma variavel categorica, revelam se subgrupos sao separaveis (util para classificacao)
- Limitacao: para $d > 10$ variaveis, pair plots ficam muito grandes; use matriz de correlacao + selecao de pares interessantes

**Conexao com 0.3:** Cada scatter plot bivariado eh uma projecao 2D dos dados. PCA (que usa autovetores de 0.3) encontra as projecoes que maximizam a variancia.

## 10. Resumo Automatico com pandas

A funcao `DataFrame.describe()` calcula automaticamente: contagem, media, desvio padrao, min, quartis e max. Combinada com `dtypes`, `isnull().sum()`, e `groupby().describe()`, oferece um panorama completo em poucas linhas.

**Por que em ML:** Estas funcoes sao o "checklist de sanidade" antes de qualquer modelagem. Valores ausentes, escalas incompativeis, tipos errados de dados -- tudo aparece aqui.

In [10]:
# Describe() fornece resumo automático
print('Resumo Estatístico (Tips Dataset):')
print(tips.describe())

print('\nResumo Detalhado (todos os percentis):')
print(tips.describe(percentiles=[0.05, 0.25, 0.5, 0.75, 0.95]))

print('\nTipos de dados:')
print(tips.dtypes)

print('\nValores ausentes:')
print(tips.isnull().sum())

print('\nDimensões:')
print(f'Linhas: {tips.shape[0]}, Colunas: {tips.shape[1]}')

# Análise descritiva por grupo
print('\nResumo por Dia da Semana:')
print(tips.groupby('day')[['total_bill', 'tip']].describe())

Resumo Estatístico (Tips Dataset):
       total_bill         tip        size
count  244.000000  244.000000  244.000000
mean    19.785943    2.998279    2.569672
std      8.902412    1.383638    0.951100
min      3.070000    1.000000    1.000000
25%     13.347500    2.000000    2.000000
50%     17.795000    2.900000    2.000000
75%     24.127500    3.562500    3.000000
max     50.810000   10.000000    6.000000

Resumo Detalhado (todos os percentis):
       total_bill         tip        size
count  244.000000  244.000000  244.000000
mean    19.785943    2.998279    2.569672
std      8.902412    1.383638    0.951100
min      3.070000    1.000000    1.000000
5%       9.557500    1.440000    2.000000
25%     13.347500    2.000000    2.000000
50%     17.795000    2.900000    2.000000
75%     24.127500    3.562500    3.000000
95%     38.061000    5.195500    4.000000
max     50.810000   10.000000    6.000000

Tipos de dados:
total_bill     float64
tip            float64
sex           category

**O que observar:**
- `describe()` mostra estatisticas por coluna: media, std, quartis
- `isnull().sum()` revela dados faltantes (critico para decidir estrategia de imputacao)
- `groupby().describe()` permite comparar estatisticas entre subgrupos

**O que concluir:**
- Sempre rode `describe()`, `dtypes`, e `isnull()` como primeiros comandos ao receber um dataset novo
- Dados faltantes precisam de tratamento: remover, imputar com media/mediana, ou usar modelos que lidam com NaN
- Comparar estatisticas entre grupos revela heterogeneidade nos dados

## 11. EDA Sistematica: Dataset Titanic

### Roteiro de EDA Profissional

1. **Estrutura:** dimensoes, tipos, valores ausentes
2. **Univariada:** distribuicao de cada variavel individualmente
3. **Bivariada:** relacao de cada feature com o target (sobrevivencia)
4. **Multivariada:** interacoes entre features
5. **Insights:** padroes principais e hipoteses

Vamos aplicar este roteiro ao dataset Titanic.

**Por que em ML:** Este roteiro eh o que todo data scientist faz antes de modelar. Para o Titanic, queremos entender que fatores influenciam a sobrevivencia -- isso guia a engenharia de features e a escolha do modelo.

In [11]:
# Estrutura básica
print('Dataset Titanic - Informações Básicas:')
print(f'Shape: {titanic.shape}')
print(f'\nTipos de dados:\n{titanic.dtypes}')
print(f'\nValores ausentes:\n{titanic.isnull().sum()}')

# Análise univariada
fig, axes = plt.subplots(2, 3, figsize=(16, 8))

# Sobrevivência (alvo)
survival_counts = titanic['survived'].value_counts()
axes[0, 0].bar(survival_counts.index, survival_counts.values, color=['lightcoral', 'lightgreen'])
axes[0, 0].set_xlabel('Sobreviveu')
axes[0, 0].set_ylabel('Contagem')
axes[0, 0].set_title(f'Distribuição: Sobrevivência\n(Taxa: {titanic["survived"].mean()*100:.1f}%)')
axes[0, 0].set_xticklabels(['Não', 'Sim'])

# Idade
axes[0, 1].hist(titanic['age'].dropna(), bins=20, edgecolor='black', alpha=0.7, color='skyblue')
axes[0, 1].set_xlabel('Idade (anos)')
axes[0, 1].set_ylabel('Frequência')
axes[0, 1].set_title('Distribuição: Idade')
axes[0, 1].axvline(titanic['age'].mean(), color='red', linestyle='--', label=f'Média: {titanic["age"].mean():.1f}')
axes[0, 1].legend()

# Classe de Passagem
class_counts = titanic['pclass'].value_counts().sort_index()
axes[0, 2].bar(class_counts.index, class_counts.values, color=['gold', 'silver', 'tan'])
axes[0, 2].set_xlabel('Classe')
axes[0, 2].set_ylabel('Contagem')
axes[0, 2].set_title('Distribuição: Classe de Passagem')

# Sexo
sex_counts = titanic['sex'].value_counts()
axes[1, 0].pie(sex_counts.values, labels=sex_counts.index, autopct='%1.1f%%', colors=['lightblue', 'lightpink'])
axes[1, 0].set_title('Distribuição: Sexo')

# Tarifa
axes[1, 1].hist(titanic['fare'].dropna(), bins=20, edgecolor='black', alpha=0.7, color='lightgreen')
axes[1, 1].set_xlabel('Tarifa ($)')
axes[1, 1].set_ylabel('Frequência')
axes[1, 1].set_title('Distribuição: Tarifa')

# Embarque
embark_counts = titanic['embarked'].value_counts()
axes[1, 2].bar(embark_counts.index, embark_counts.values, color='lightyellow', edgecolor='black')
axes[1, 2].set_xlabel('Porto de Embarque')
axes[1, 2].set_ylabel('Contagem')
axes[1, 2].set_title('Distribuição: Porto de Embarque')

plt.tight_layout()
plt.show()

# Bivariada: Sobrevivência vs Outras Variáveis
fig, axes = plt.subplots(1, 3, figsize=(16, 4))

# Sobrevivência vs Sexo
sns.countplot(data=titanic, x='sex', hue='survived', ax=axes[0])
axes[0].set_title('Sobrevivência por Sexo')
axes[0].set_xlabel('Sexo')
axes[0].legend(title='Sobreviveu', labels=['Não', 'Sim'])

# Sobrevivência vs Classe
sns.countplot(data=titanic, x='pclass', hue='survived', ax=axes[1])
axes[1].set_title('Sobrevivência por Classe')
axes[1].set_xlabel('Classe')
axes[1].legend(title='Sobreviveu', labels=['Não', 'Sim'])

# Idade vs Sobrevivência
sns.boxplot(data=titanic, x='survived', y='age', ax=axes[2])
axes[2].set_xticklabels(['Não', 'Sim'])
axes[2].set_xlabel('Sobreviveu')
axes[2].set_ylabel('Idade')
axes[2].set_title('Distribuição de Idade por Sobrevivência')

plt.tight_layout()
plt.show()

Dataset Titanic - Informações Básicas:
Shape: (891, 15)

Tipos de dados:
survived          int64
pclass            int64
sex                 str
age             float64
sibsp             int64
parch             int64
fare            float64
embarked            str
class          category
who                 str
adult_male         bool
deck           category
embark_town         str
alive               str
alone              bool
dtype: object

Valores ausentes:
survived         0
pclass           0
sex              0
age            177
sibsp            0
parch            0
fare             0
embarked         2
class            0
who              0
adult_male       0
deck           688
embark_town      2
alive            0
alone            0
dtype: int64


**O que observar:**
- Taxa de sobrevivencia geral: ~38% (desbalanceado -- maioria nao sobreviveu)
- Distribuicao de idade: assimetrica positiva, com pico em 20-30 anos
- Classe 3 tem mais passageiros, mas classe 1 tem maior taxa de sobrevivencia
- Mulheres tiveram taxa de sobrevivencia muito maior que homens ("mulheres e criancas primeiro")
- Idade vs sobrevivencia: distribuicoes similares, mas criancas tinham vantagem

**O que concluir:**
- Sexo e classe sao os preditores mais fortes de sobrevivencia (fariam boas features em um modelo)
- A tarifa tem distribuicao muito assimetrica com outliers -- candidata a transformacao log
- Dados faltantes em "age" precisam de imputacao (mediana por classe ou modelo)
- Um modelo simples (sexo + classe) ja teria acuracia razoavel

**Conexao com 0.8:** Se fossemos treinar um modelo de classificacao aqui, usariamos cross-entropy como funcao de custo (classificacao binaria) e Adam como otimizador -- exatamente o que aprendemos em 0.8.

## 12. Exercicios Praticos

### Exercicio 1: Analise Descritiva Completa do Titanic
Calcule todas as medidas descritivas para 'age' e 'fare': media, mediana, desvio padrao, IQR, skewness, kurtosis. Identifique outliers pelo metodo IQR. Compare as estatisticas entre sobreviventes e nao-sobreviventes.

**Dica:** use `groupby('survived')` para separar os grupos.

In [12]:
# EXERCICIO 1: Analise Descritiva do Titanic
# Complete o codigo abaixo

def analise_descritiva(df, coluna, grupo=None):
    """
    Calcula medidas descritivas completas para uma coluna.
    Se grupo for fornecido, calcula por grupo.

    Retorna: dict com media, mediana, std, iqr, skewness, kurtosis, n_outliers
    """
    if grupo:
        resultados = {}
        for name, group_df in df.groupby(grupo):
            data = group_df[coluna].dropna()
            resultados[name] = {
                'media': None,     # TAREFA DO ALUNO
                'mediana': None,   # TAREFA DO ALUNO
                'std': None,       # TAREFA DO ALUNO
                'iqr': None,       # TAREFA DO ALUNO: Q3 - Q1
                'skewness': None,  # TAREFA DO ALUNO: usar stats.skew()
                'kurtosis': None,  # TAREFA DO ALUNO: usar stats.kurtosis()
                'n_outliers': None # TAREFA DO ALUNO: contar outliers IQR
            }
        return resultados
    else:
        data = df[coluna].dropna()
        return {
            'media': None,     # TAREFA DO ALUNO
            'mediana': None,   # TAREFA DO ALUNO
            'std': None,       # TAREFA DO ALUNO
        }

# Teste (descomentar apos implementar):
# result = analise_descritiva(titanic, 'age', grupo='survived')
# for surv, stats_dict in result.items():
#     print(f"Survived={surv}: {stats_dict}")

In [13]:
# SOLUCAO Exercicio 1

def analise_descritiva(df, coluna, grupo=None):
    def calc_stats(data):
        Q1, Q3 = data.quantile(0.25), data.quantile(0.75)
        IQR = Q3 - Q1
        outliers = data[(data < Q1 - 1.5*IQR) | (data > Q3 + 1.5*IQR)]
        return {
            'n': len(data),
            'media': data.mean(),
            'mediana': data.median(),
            'std': data.std(),
            'iqr': IQR,
            'skewness': stats.skew(data),
            'kurtosis': stats.kurtosis(data),
            'n_outliers': len(outliers)
        }

    if grupo:
        resultados = {}
        for name, group_df in df.groupby(grupo):
            resultados[name] = calc_stats(group_df[coluna].dropna())
        return resultados
    else:
        return calc_stats(df[coluna].dropna())

print("=== ANALISE DESCRITIVA: TITANIC ===\n")
for col in ['age', 'fare']:
    print(f"--- {col.upper()} ---")
    result = analise_descritiva(titanic, col, grupo='survived')
    for surv, s in result.items():
        label = "Sobreviveu" if surv == 1 else "Nao sobreviveu"
        print(f"  {label}: media={s['media']:.1f}, mediana={s['mediana']:.1f}, "
              f"std={s['std']:.1f}, IQR={s['iqr']:.1f}, "
              f"skew={s['skewness']:.2f}, outliers={s['n_outliers']}")
    print()

print("Insight: Sobreviventes pagaram tarifa media MAIOR (classe mais alta)")
print("Insight: Idade media similar entre grupos, mas distribuicao pode diferir")

=== ANALISE DESCRITIVA: TITANIC ===

--- AGE ---
  Nao sobreviveu: media=30.6, mediana=28.0, std=14.2, IQR=18.0, skew=0.58, outliers=6
  Sobreviveu: media=28.3, mediana=28.0, std=15.0, IQR=17.0, skew=0.18, outliers=5

--- FARE ---
  Nao sobreviveu: media=22.1, mediana=10.5, std=31.4, IQR=18.1, skew=4.54, outliers=43
  Sobreviveu: media=48.4, mediana=26.0, std=66.6, IQR=44.5, skew=3.85, outliers=28

Insight: Sobreviventes pagaram tarifa media MAIOR (classe mais alta)
Insight: Idade media similar entre grupos, mas distribuicao pode diferir


### Exercicio 2: Deteccao de Outliers Multi-metodo
Implemente os 3 metodos de deteccao de outliers (IQR, Z-score, MAD) para a coluna 'fare' do Titanic. Compare quantos outliers cada metodo detecta e visualize os resultados.

**Dica:** para MAD, use `np.median(np.abs(data - np.median(data)))` e multiplique por 1.4826 para estimar $\sigma$.

In [14]:
# EXERCICIO 2: Deteccao de Outliers Multi-metodo
# Complete o codigo abaixo

def detectar_outliers(data, metodo='iqr'):
    """
    Detecta outliers usando IQR, Z-score ou MAD.
    Retorna: mask booleano (True = outlier)
    """
    if metodo == 'iqr':
        Q1, Q3 = data.quantile(0.25), data.quantile(0.75)
        IQR = Q3 - Q1
        mask = None  # TAREFA DO ALUNO: (data < Q1 - 1.5*IQR) | (data > Q3 + 1.5*IQR)
    elif metodo == 'zscore':
        z = None  # TAREFA DO ALUNO: (data - data.mean()) / data.std()
        mask = None  # TAREFA DO ALUNO: np.abs(z) > 3
    elif metodo == 'mad':
        median = data.median()
        mad = None  # TAREFA DO ALUNO: np.median(np.abs(data - median)) * 1.4826
        mask = None  # TAREFA DO ALUNO: np.abs(data - median) / mad > 3

    return mask

# Teste (descomentar apos implementar):
# for m in ['iqr', 'zscore', 'mad']:
#     mask = detectar_outliers(titanic['fare'].dropna(), m)
#     print(f"{m}: {mask.sum()} outliers")

In [15]:
# SOLUCAO Exercicio 2

def detectar_outliers(data, metodo='iqr'):
    if metodo == 'iqr':
        Q1, Q3 = data.quantile(0.25), data.quantile(0.75)
        IQR = Q3 - Q1
        mask = (data < Q1 - 1.5*IQR) | (data > Q3 + 1.5*IQR)
    elif metodo == 'zscore':
        z = (data - data.mean()) / data.std()
        mask = np.abs(z) > 3
    elif metodo == 'mad':
        median = data.median()
        mad = np.median(np.abs(data - median)) * 1.4826
        mask = np.abs(data - median) / mad > 3
    return mask

fare = titanic['fare'].dropna()

print("Deteccao de outliers em FARE:\n")
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

for idx, metodo in enumerate(['iqr', 'zscore', 'mad']):
    mask = detectar_outliers(fare, metodo)
    n_out = mask.sum()

    ax = axes[idx]
    ax.scatter(range(len(fare)), fare.values, alpha=0.4, s=20, label='Normal')
    ax.scatter(fare.index[mask], fare[mask].values, color='red', s=40, label=f'Outliers ({n_out})')
    ax.set_xlabel('Indice')
    ax.set_ylabel('Tarifa ($)')
    ax.set_title(f'{metodo.upper()}: {n_out} outliers ({n_out/len(fare)*100:.1f}%)')
    ax.legend()
    ax.grid(alpha=0.3)

plt.tight_layout()
plt.savefig('/tmp/ex2_outliers.png', dpi=100, bbox_inches='tight')
plt.close()

print(f"{'Metodo':>8} | {'Outliers':>8} | {'%':>6}")
print("-" * 30)
for m in ['iqr', 'zscore', 'mad']:
    mask = detectar_outliers(fare, m)
    print(f"{m:>8} | {mask.sum():>8} | {mask.sum()/len(fare)*100:>5.1f}%")

print("\nMAD detecta mais outliers que Z-score (mais conservador)")
print("IQR e MAD sao robustos; Z-score eh puxado pelos proprios outliers")

Deteccao de outliers em FARE:



  Metodo | Outliers |      %
------------------------------


     iqr |      116 |  13.0%
  zscore |       20 |   2.2%
     mad |      171 |  19.2%

MAD detecta mais outliers que Z-score (mais conservador)
IQR e MAD sao robustos; Z-score eh puxado pelos proprios outliers


### Exercicio 3: Pair Plot com Insights
Usando o dataset Penguins, crie um pair plot colorido por especie. Identifique o par de variaveis com maior correlacao e o par com menor correlacao. Para cada especie, calcule a media de body_mass_g.

**Dica:** use `penguins.groupby('species')[colunas].corr()` para correlacoes por especie.

In [16]:
# EXERCICIO 3: Pair Plot com Insights
# Complete o codigo abaixo

numeric_cols = ['bill_length_mm', 'bill_depth_mm', 'flipper_length_mm', 'body_mass_g']

# TAREFA DO ALUNO: Calcular matriz de correlacao
# corr = penguins[numeric_cols].corr()

# TAREFA DO ALUNO: Encontrar par com maior correlacao (excluindo diagonal)
# Dica: use corr.unstack() e filtre valores != 1.0

# TAREFA DO ALUNO: Encontrar par com menor correlacao

# TAREFA DO ALUNO: Media de body_mass_g por especie
# print(penguins.groupby('species')['body_mass_g'].mean())

In [17]:
# SOLUCAO Exercicio 3

numeric_cols = ['bill_length_mm', 'bill_depth_mm', 'flipper_length_mm', 'body_mass_g']

# Matriz de correlacao
corr = penguins[numeric_cols].corr()
print("Matriz de Correlacao:")
print(corr.round(3))

# Encontrar pares extremos
corr_unstacked = corr.unstack()
# Remover diagonal e duplicatas
mask = np.triu(np.ones_like(corr, dtype=bool), k=1)
corr_pairs = corr.where(mask).unstack().dropna()

max_pair = corr_pairs.idxmax()
min_pair = corr_pairs.idxmin()
print(f"\nMaior correlacao: {max_pair[0]} vs {max_pair[1]} = {corr_pairs.max():.3f}")
print(f"Menor correlacao: {min_pair[0]} vs {min_pair[1]} = {corr_pairs.min():.3f}")

# Media por especie
print("\nMassa corporal media por especie:")
print(penguins.groupby('species')['body_mass_g'].agg(['mean', 'std', 'count']).round(1))

# Pair plot
fig = sns.pairplot(penguins[numeric_cols + ['species']], hue='species',
                   diag_kind='kde', plot_kws={'alpha': 0.6})
plt.savefig('/tmp/ex3_pairplot.png', dpi=80, bbox_inches='tight')
plt.close()

print("\nInsights:")
print("- Flipper length e body mass tem correlacao mais forte (pinguins maiores tem nadadeiras maiores)")
print("- Bill depth tem correlacao NEGATIVA com flipper length (paradoxo de Simpson!)")
print("- Gentoo sao claramente maiores e mais pesados que as outras especies")
print("- Adelie e Chinstrap tem tamanhos similares mas bicos diferentes")

Matriz de Correlacao:
                   bill_length_mm  bill_depth_mm  flipper_length_mm  \
bill_length_mm              1.000         -0.229              0.653   
bill_depth_mm              -0.229          1.000             -0.578   
flipper_length_mm           0.653         -0.578              1.000   
body_mass_g                 0.589         -0.472              0.873   

                   body_mass_g  
bill_length_mm           0.589  
bill_depth_mm           -0.472  
flipper_length_mm        0.873  
body_mass_g              1.000  

Maior correlacao: body_mass_g vs flipper_length_mm = 0.873
Menor correlacao: flipper_length_mm vs bill_depth_mm = -0.578

Massa corporal media por especie:
             mean    std  count
species                        
Adelie     3706.2  458.6    146
Chinstrap  3733.1  384.3     68
Gentoo     5092.4  501.5    119



Insights:
- Flipper length e body mass tem correlacao mais forte (pinguins maiores tem nadadeiras maiores)
- Bill depth tem correlacao NEGATIVA com flipper length (paradoxo de Simpson!)
- Gentoo sao claramente maiores e mais pesados que as outras especies
- Adelie e Chinstrap tem tamanhos similares mas bicos diferentes


## 13. Erros Comuns

### Erro 1: Usar Media em Dados com Outliers Severos
**Sintoma:** a media nao "parece" representativa dos dados
**Causa:** poucos valores extremos puxam a media desproporcionalmente
**Solucao:** usar mediana ou media aparada (trimmed mean); investigar os outliers

### Erro 2: Ignorar a Forma da Distribuicao
**Sintoma:** duas distribuicoes com mesma media e desvio padrao parecem iguais, mas graficamente sao muito diferentes
**Causa:** media e desvio padrao nao capturam assimetria, bimodalidade, caudas pesadas
**Solucao:** SEMPRE visualizar os dados; usar skewness/kurtosis; olhar QQ-plot

### Erro 3: Confundir Desvio Padrao com Erro Padrao
**Sintoma:** reportar incerteza da media usando desvio padrao
**Causa:** $s$ mede dispersao dos dados; $s/\sqrt{n}$ (erro padrao) mede incerteza da estimativa da media
**Solucao:** para dispersao dos dados, use $s$; para incerteza da media, use $s/\sqrt{n}$

### Erro 4: Usar Pearson para Relacoes Nao-Lineares
**Sintoma:** correlacao de Pearson perto de zero, mas scatter plot mostra relacao clara
**Causa:** Pearson mede apenas relacao LINEAR; parabolas ou curvas em U dao $r \approx 0$
**Solucao:** usar Spearman (baseado em rankings) ou inspecionar scatter plots

### Erro 5: Correlacao Implica Causalidade
**Sintoma:** concluir que A causa B porque estao correlacionados
**Causa:** confundir associacao com mecanismo causal; ignorar variaveis confundidoras
**Solucao:** lembrar que correlacao pode vir de: A causa B, B causa A, C causa ambos, ou coincidencia

### Erro 6: Nao Verificar Pressupostos dos Testes
**Sintoma:** teste estatistico "funciona" (roda sem erro) mas da resultado errado
**Causa:** usar teste parametrico (assume normalidade) em dados nao-normais
**Solucao:** checar normalidade (QQ-plot, Shapiro-Wilk) antes de usar testes parametricos; usar alternativas nao-parametricas quando necessario

### Erro 7: Analisar Dados sem Tratar Valores Ausentes
**Sintoma:** resultados viesados ou funcoes retornando NaN
**Causa:** valores ausentes podem ser sistematicos (MNAR) e ignorar isso enviesa a analise
**Solucao:** documentar padroes de dados faltantes; decidir estrategia (remover, imputar) ANTES de analisar

## 14. Resumo e Mapa Conceitual

### Hierarquia da Analise Descritiva

```
DADOS BRUTOS
    |
    v
UNIVARIADA (1 variavel por vez)
    |--- Tendencia Central: Media, Mediana, Moda
    |--- Dispersao: Desvio Padrao, IQR, CV, MAD
    |--- Forma: Skewness, Kurtosis
    |--- Outliers: IQR, Z-score, MAD
    |--- Visualizar: Histograma, KDE, Boxplot, Violin
    |
    v
BIVARIADA (2 variaveis)
    |--- Numerica x Numerica: Pearson, Spearman, Scatter
    |--- Categorica x Categorica: Chi-quadrado, Contingencia
    |--- Numerica x Categorica: Boxplot por grupo, T-test
    |--- Visualizar: Scatter, Heatmap, Violin por grupo
    |
    v
MULTIVARIADA (3+ variaveis)
    |--- Pair plots: todas as relacoes bivariadas
    |--- Matriz de correlacao: visao geral
    |--- PCA: reducao de dimensionalidade (futuro)
    |
    v
INSIGHTS para ML
    |--- Quais features usar?
    |--- Precisa normalizar?
    |--- Outliers: remover ou tratar?
    |--- Dados faltantes: qual estrategia?
```

### Tabela de Conexoes

| Conceito deste notebook | Fundamento matematico | Notebook |
|-------------------------|----------------------|----------|
| Media amostral | $E[X]$ (esperanca) | 0.6 |
| Desvio padrao | $\sqrt{\text{Var}(X)}$ | 0.6 |
| Correlacao de Pearson | $\text{Cov}(X,Y) / (\sigma_X \sigma_Y)$ | 0.7 |
| Media como estimador | MLE para $\mu$ gaussiano | 0.7 |
| Media vs mediana | MSE vs MAE como custo | 0.8 |
| QQ-plot | Quantis de distribuicao normal | 0.6 |
| Normalizacao de features | $z = (x-\mu)/\sigma$ (z-score) | 0.6 |
| Deteccao de outliers IQR | Quartis da distribuicao | 0.6 |

### Checklist de Competencias

Apos completar este modulo, voce deve ser capaz de:

- [ ] Calcular e interpretar media, mediana, moda em datasets reais
- [ ] Escolher medida de tendencia central adequada (com vs sem outliers)
- [ ] Calcular e interpretar desvio padrao, IQR, CV
- [ ] Avaliar normalidade usando skewness, kurtosis e QQ-plots
- [ ] Detectar outliers com IQR, Z-score e MAD
- [ ] Visualizar distribuicoes com histogramas, KDE, boxplots e violin plots
- [ ] Calcular e testar correlacoes (Pearson e Spearman)
- [ ] Analisar associacao entre variaveis categoricas (qui-quadrado)
- [ ] Realizar EDA sistematica em datasets novos
- [ ] Gerar insights acionaveis para feature engineering e modelagem

### Proximos Passos

- **1.2 (Estatistica Inferencial):** Formalizar como usar amostras para fazer afirmacoes sobre populacoes (intervalos de confianca, testes de hipotese)
- **1.3 (Estatistica Bayesiana):** Incorporar conhecimento previo na analise
- **1.4 (Regressao Estatistica):** Modelar relacoes entre variaveis (conecta com correlacao deste notebook)